# MedXAI — Phase 2 : Fine-tuning Bio_ClinicalBERT
## Classification multi-label de maladies respiratoires avec un LLM médical

**Objectif :** Fine-tuner Bio_ClinicalBERT sur nos données de symptômes respiratoires
et comparer ses performances avec le XGBoost de la Phase 1.

**Modèle :** emilyalsentzer/Bio_ClinicalBERT — pré-entraîné sur des notes cliniques médicales

**Maladies :** Bronchial Asthma · Tuberculosis · Pneumonia · Common Cold

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# HuggingFace
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import torch

# Sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


sys.path.append(os.path.abspath('..'))

print(f"Imports OK")
print(f"PyTorch version : {torch.__version__}")
print(f"GPU disponible : {torch.cuda.is_available()}")

Imports OK
PyTorch version : 2.13.0+cpu
GPU disponible : False


In [2]:
# Charger le dataset
df = pd.read_csv('data/dataset.csv')

# Filtrer les maladies respiratoires
maladies = ['Bronchial Asthma', 'Tuberculosis', 'Pneumonia', 'Common Cold']
df_resp = df[df['Disease'].isin(maladies)].reset_index(drop=True)

# Convertir les symptômes en texte naturel
symptom_cols = [c for c in df_resp.columns if c.startswith('Symptom_')]

def symptoms_to_text(row):
    symptoms = [row[col].strip() for col in symptom_cols if pd.notna(row[col])]
    return f"Patient presents with : {', '.join(symptoms)}"

df_resp['text'] = df_resp.apply(symptoms_to_text, axis=1)

le = LabelEncoder()
df_resp['label'] = le.fit_transform(df_resp['Disease'])

print(f"Dataset prêt : {len(df_resp)} patients")
print(f"\nExemple de texte généré :")
print(df_resp['text'].iloc[0])
print(f"\nLabel : {df_resp['Disease'].iloc[0]} : {df_resp['label'].iloc[0]}")

Dataset prêt : 480 patients

Exemple de texte généré :
Patient presents with : fatigue, cough, high_fever, breathlessness, family_history, mucoid_sputum

Label : Bronchial Asthma : 0


In [3]:

train_df, test_df = train_test_split(
    df_resp[['text', 'label']],
    test_size=0.2,
    random_state=42,
    stratify=df_resp['label']
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train : {len(train_df)} patients")
print(f"Test  : {len(test_df)} patients")

# Convertir en Dataset HuggingFace
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(f"\n Datasets HuggingFace créés")
print(train_dataset)

Train : 384 patients
Test  : 96 patients

 Datasets HuggingFace créés
Dataset({
    features: ['text', 'label'],
    num_rows: 384
})


In [4]:

model_name = "emilyalsentzer/Bio_ClinicalBERT"

print(f"Chargement du tokenizer : {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenizer le dataset
def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

train_tokenized = train_dataset.map(tokenize, batched=True)
test_tokenized = test_dataset.map(tokenize, batched=True)

print("Tokenization terminée")
print(f"\nExemple de tokens pour le premier patient :")
print(f"Texte : {train_df['text'].iloc[0]}")
print(f"Nb tokens : {len(tokenizer(train_df['text'].iloc[0])['input_ids'])}")

Chargement du tokenizer : emilyalsentzer/Bio_ClinicalBERT
(Premier téléchargement — peut prendre 1-2 minutes...)



Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Tokenization terminée

Exemple de tokens pour le premier patient :
Texte : Patient presents with : chills, fatigue, cough, high_fever, breathlessness, sweating, malaise, chest_pain, fast_heart_rate, rusty_sputum
Nb tokens : 41


In [5]:
print("Chargement du modèle Bio_ClinicalBERT...")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(le.classes_)
)

print(f" Modèle chargé")
print(f"Nombre de classes : {len(le.classes_)}")
print(f"Classes : {le.classes_}")

Chargement du modèle Bio_ClinicalBERT...
(Téléchargement en cours — 400MB environ...)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

 Modèle chargé
Nombre de classes : 4
Classes : ['Bronchial Asthma' 'Common Cold' 'Pneumonia' 'Tuberculosis']


In [7]:

# FULL FINE-TUNING Bio_ClinicalBERT

from transformers import DataCollatorWithPadding
import evaluate

# Metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = (predictions == labels).mean()
    return {"accuracy": acc}

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Arguments d'entraînement
training_args = TrainingArguments(
    output_dir='../results/clinical_bert',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir='../results/logs',
    logging_steps=10,
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer configuré")
print(f"Epochs : 3")
print(f"Batch size : 8")
print(f"Données train : {len(train_tokenized)}")
print("Lancement du fine-tuning...")
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer configuré
Epochs : 3
Batch size : 8
Données train : 384
Lancement du fine-tuning...


C:\Users\Rep Tech\Desktop\MedXAI-ACF-Anchor\medxai-env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.064791,0.013878,1.000000
2,0.005381,0.003386,1.000000
3,0.003863,0.002703,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Rep Tech\Desktop\MedXAI-ACF-Anchor\medxai-env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Rep Tech\Desktop\MedXAI-ACF-Anchor\medxai-env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=144, training_loss=0.1564243339234963, metrics={'train_runtime': 1189.8365, 'train_samples_per_second': 0.968, 'train_steps_per_second': 0.121, 'total_flos': 75777344667648.0, 'train_loss': 0.1564243339234963, 'epoch': 3.0})

In [8]:

# ÉVALUATION DU MODÈLE

# Prédictions sur le test set
predictions = trainer.predict(test_tokenized)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Rapport de classification
print(" Rapport de classification Bio_ClinicalBERT :\n")
print(classification_report(y_true, y_pred, target_names=le.classes_))

C:\Users\Rep Tech\Desktop\MedXAI-ACF-Anchor\medxai-env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


 Rapport de classification Bio_ClinicalBERT :

                  precision    recall  f1-score   support

Bronchial Asthma       1.00      1.00      1.00        24
     Common Cold       1.00      1.00      1.00        24
       Pneumonia       1.00      1.00      1.00        24
    Tuberculosis       1.00      1.00      1.00        24

        accuracy                           1.00        96
       macro avg       1.00      1.00      1.00        96
    weighted avg       1.00      1.00      1.00        96



In [9]:

# SAUVEGARDER LE MODÈLE FINE-TUNÉ

model.save_pretrained('../results/clinical_bert/final_model')
tokenizer.save_pretrained('../results/clinical_bert/final_model')

print(" Modèle sauvegardé dans results/clinical_bert/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Modèle sauvegardé dans results/clinical_bert/final_model
